# Fase 2 — Data Understanding
## Construcción y validación de DENUE 2025

### Proyecto
Modelo econométrico de atractividad comercial municipal en México.

### Objetivo de esta etapa
Explorar y validar los archivos DENUE correspondientes al comercio al por menor en 2025 antes de construir la base municipal.

En esta etapa se verificará:

- disponibilidad de los archivos;
- estructura interna de los archivos ZIP;
- nombres de los archivos CSV;
- número de registros;
- variables disponibles;
- tipos de datos;
- códigos de actividad económica;
- claves de entidad y municipio;
- valores faltantes;
- duplicados;
- cobertura municipal;
- consistencia estructural para su posterior comparación con DENUE 2020.

Todavía no se realizarán regresiones econométricas.

In [1]:
#################3. Librerías
from pathlib import Path
import zipfile

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


In [2]:
################4. Definir ruta DENUE 2025
# Ruta raíz del proyecto
PROJECT_ROOT = Path.cwd().parent

# Ruta de los archivos DENUE 2025
DENUE_2025_DIR = PROJECT_ROOT / "data" / "raw" / "denue" / "2025"

print("Ruta raíz del proyecto:")
print(PROJECT_ROOT)

print("\nRuta DENUE 2025:")
print(DENUE_2025_DIR)

print("\n¿Existe la carpeta?:", DENUE_2025_DIR.exists())

Ruta raíz del proyecto:
c:\Users\nashe\Desktop\Econometría\PROYECTO\atractividad_comercial_Mexico

Ruta DENUE 2025:
c:\Users\nashe\Desktop\Econometría\PROYECTO\atractividad_comercial_Mexico\data\raw\denue\2025

¿Existe la carpeta?: True


In [3]:
###################5. Detectar automáticamente los cuatro ZIP
archivos_zip_2025 = sorted(DENUE_2025_DIR.glob("*.zip"))

print(
    f"Número de archivos ZIP encontrados: "
    f"{len(archivos_zip_2025)}\n"
)

for archivo in archivos_zip_2025:
    print(archivo.name)

Número de archivos ZIP encontrados: 4

denue_00_46111_0525_csv.zip
denue_00_46112-46311_0525_csv.zip
denue_00_46321-46531_0525_csv.zip
denue_00_46591-46911_0525_csv.zip


6. Revisar la estructura interna
### 2.1 Inspección de la estructura interna de los archivos DENUE 2025

Se inspecciona el contenido de los cuatro paquetes DENUE 2025 antes de cargar los datos.

El objetivo es identificar:

- archivo principal de datos;
- diccionario de datos;
- metadatos;
- estructura de carpetas.

Esta revisión permitirá comprobar posteriormente si DENUE 2025 presenta una estructura compatible con DENUE 2020.

In [4]:
for archivo_zip in archivos_zip_2025:

    print("=" * 90)
    print(f"ARCHIVO: {archivo_zip.name}")
    print("=" * 90)

    with zipfile.ZipFile(archivo_zip, "r") as z:

        for elemento in z.namelist():
            print(elemento)

    print()

ARCHIVO: denue_00_46111_0525_csv.zip
diccionario_de_datos/denue_diccionario_de_datos.csv
conjunto_de_datos/denue_inegi_46111_.csv
metadatos/metadatos_denue.txt

ARCHIVO: denue_00_46112-46311_0525_csv.zip
diccionario_de_datos/denue_diccionario_de_datos.csv
conjunto_de_datos/denue_inegi_46112-46311_.csv
metadatos/metadatos_denue.txt

ARCHIVO: denue_00_46321-46531_0525_csv.zip
diccionario_de_datos/denue_diccionario_de_datos.csv
conjunto_de_datos/denue_inegi_46321-46531_.csv
metadatos/metadatos_denue.txt

ARCHIVO: denue_00_46591-46911_0525_csv.zip
diccionario_de_datos/denue_diccionario_de_datos.csv
conjunto_de_datos/denue_inegi_46591-46911_.csv
metadatos/metadatos_denue.txt



### 2.2 Lectura controlada de los archivos principales

Antes de cargar completamente los archivos DENUE 2025, se realiza una lectura de muestra de cada conjunto de datos.

El objetivo es verificar:

- número de columnas;
- nombres de las variables;
- estructura de los registros;
- disponibilidad de las claves geográficas;
- disponibilidad de los códigos de actividad económica;
- consistencia estructural entre los cuatro archivos.

Para mantener un uso controlado de memoria, en esta etapa se leen únicamente las primeras cinco filas de cada archivo.

In [5]:
# Diccionario para almacenar muestras DENUE 2025
muestras_denue_2025 = {}

for archivo_zip in archivos_zip_2025:

    with zipfile.ZipFile(archivo_zip, "r") as z:

        # Identificar automáticamente el CSV principal
        archivos_datos = [
            nombre for nombre in z.namelist()
            if nombre.startswith("conjunto_de_datos/")
            and nombre.endswith(".csv")
        ]

        if len(archivos_datos) != 1:
            print(
                f"Advertencia en {archivo_zip.name}: "
                f"se encontraron {len(archivos_datos)} archivos de datos."
            )
            continue

        csv_principal = archivos_datos[0]

        # Leer únicamente las primeras 5 filas
        with z.open(csv_principal) as archivo_csv:

            muestra = pd.read_csv(
                archivo_csv,
                nrows=5,
                encoding="latin-1"
            )

        muestras_denue_2025[archivo_zip.name] = muestra

        print("=" * 100)
        print(f"ZIP: {archivo_zip.name}")
        print(f"CSV: {csv_principal}")
        print(f"Número de columnas: {muestra.shape[1]}")
        print("=" * 100)

        display(muestra)

ZIP: denue_00_46111_0525_csv.zip
CSV: conjunto_de_datos/denue_inegi_46111_.csv
Número de columnas: 42


,id,clee,nom_estab,raz_social,codigo_act,nombre_act,per_ocu,tipo_vial,nom_vial,tipo_v_e_1,nom_v_e_1,tipo_v_e_2,nom_v_e_2,tipo_v_e_3,nom_v_e_3,numero_ext,letra_ext,edificio,edificio_e,numero_int,letra_int,tipo_asent,nomb_asent,tipoCenCom,nom_CenCom,num_local,cod_postal,cve_ent,entidad,cve_mun,municipio,cve_loc,localidad,ageb,manzana,telefono,correoelec,www,tipoUniEco,latitud,longitud,fecha_alta
0,9604852,01001461110069191000064636M1,2BOX,ALPEZ NBG SA DE CV,461110,"Comercio al por menor en tiendas de abarrotes,...",0 a 5 personas,AVENIDA,QUINTA,CALLE,REPUBLICA DE GUATEMALA,AVENIDA,CONVENCION DE 1914 SUR,CALLE,REPUBLICA DE ARGENTINA,602,NaN,NaN,NaN,NaN,NaN,FRACCIONAMIENTO,LAS AMERICAS,NaN,NaN,NaN,20230,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,869,19,3.339524e+09,CAPAGAS@2BOX.MX,NaN,Fijo,21.865466,-102.298267,2024-11
1,10060248,01001461110065211000000000S1,2BOX,2BOX SA DE CV,461110,"Comercio al por menor en tiendas de abarrotes,...",0 a 5 personas,BOULEVARD,BOULEVARD A ZACATECAS,CALLE,NINGUNO,AVENIDA,HEROE DE NACOZARI NORTE,CALLE,NINGUNO,605,NaN,NaN,NaN,NaN,NaN,FRACCIONAMIENTO,LAS HADAS,NaN,NaN,NaN,20140,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1975,2,NaN,NaN,NaN,Fijo,21.911625,-102.292004,2024-11
2,10550501,01001461110065751000000000U6,3 CACHORROS,NaN,461110,"Comercio al por menor en tiendas de abarrotes,...",0 a 5 personas,CALLE,OSCAR HERNANDEZ DUQUE MEDINA,CALLE,JESUS DURON,CALLE,BERNARDINO ALVAREZ,CALLE,FELIPE COSIO GUTIERREZ DE VELAZCO,247,NaN,NaN,NaN,NaN,NaN,FRACCIONAMIENTO,VIÑEDOS DEL SUR,NaN,NaN,NaN,20298,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1161,26,NaN,NaN,NaN,Fijo,21.806671,-102.272522,2024-11
3,6988165,01001461130012911000000000U8,ABAROTES DANI,NaN,461110,"Comercio al por menor en tiendas de abarrotes,...",0 a 5 personas,CALLE,MARCOS CORONADO,CALLE,SANTA MONICA,CALLE,LEONARDO MURO LOPEZ,CALLE,MARCOS CORONADO,248,NaN,NaN,NaN,0.0,NaN,EJIDO,LOS POCITOS,NaN,NaN,NaN,20328,1,Aguascalientes,1,Aguascalientes,1025,Pocitos ...,3948,34,NaN,NaN,NaN,Fijo,21.927296,-102.335457,2019-11
4,11307693,01001461110064081000000000U9,ABAROTES LULA,NaN,461110,"Comercio al por menor en tiendas de abarrotes,...",0 a 5 personas,CALLE,SOBERANA CONVENCION MILITAR REVOLUCIONARIA SUR,CALLE,GENERAL PASCUAL CORNEJO BRUN,CALLE,JOSE CALVILLO,CALLE,JOSE CALVILLO,0,SN,NaN,NaN,NaN,NaN,FRACCIONAMIENTO,JOSE LOPEZ PORTILLO,NaN,NaN,NaN,20206,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1392,13,4.496902e+09,NaN,NaN,Fijo,21.851601,-102.325927,2024-11


ZIP: denue_00_46112-46311_0525_csv.zip
CSV: conjunto_de_datos/denue_inegi_46112-46311_.csv
Número de columnas: 42


,id,clee,nom_estab,raz_social,codigo_act,nombre_act,per_ocu,tipo_vial,nom_vial,tipo_v_e_1,nom_v_e_1,tipo_v_e_2,nom_v_e_2,tipo_v_e_3,nom_v_e_3,numero_ext,letra_ext,edificio,edificio_e,numero_int,letra_int,tipo_asent,nomb_asent,tipoCenCom,nom_CenCom,num_local,cod_postal,cve_ent,entidad,cve_mun,municipio,cve_loc,localidad,ageb,manzana,telefono,correoelec,www,tipoUniEco,latitud,longitud,fecha_alta
0,11288711,01006461140000061000000000U4,@GRANEL,NaN,461140,Comercio al por menor de semillas y granos ali...,0 a 5 personas,CALLE,JUPITER,CALLE,ALTA,CALLE,ANTARES,CALLE,ORION,223,NaN,NaN,NaN,NaN,NaN,FRACCIONAMIENTO,COSMOS,NaN,NaN,NaN,20676,1,Aguascalientes,6,Pabellón de Arteaga,1,Pabellón de Arteaga ...,282,20,4.651166e+09,NaN,NaN,Fijo,22.137327,-102.279964,2024-11
1,6281207,01001463111000223001004471S0,028 MODATELAS AGUASCALIENTES I,MODATELAS SAPI DE CV,463111,Comercio al por menor de telas,6 a 10 personas,PEATONAL,BENITO JUÁREZ,AVENIDA,FRANCISCO I MADERO,PEATONAL,ALLENDE ORIENTE,CALLE,JOSÉ MARÍA MORELOS Y PAVÓN,109,NaN,NaN,NaN,NaN,NaN,COLONIA,CENTRO,NaN,NaN,NaN,20000,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,638,8,NaN,TIENDAENLINEA@MODATELAS.COM.MX,WWW.MODATELAS.COM,Fijo,21.881823,-102.295760,2010-07
2,6281091,01001463111000264001004471S4,035 MODATELAS AGUASCALIENTES II,MODATELAS SAPI DE CV,463111,Comercio al por menor de telas,6 a 10 personas,CALLE,RIVERO Y GUTIÉRREZ,CALLE,LICENCIADO BENITO JUÁREZ,PASAJE,ORTEGA,PASAJE,SAN ANTONIO,121,NaN,NaN,NaN,NaN,NaN,COLONIA,CENTRO,NaN,NaN,NaN,20000,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,515,45,NaN,TIENDAENLINEA@MODATELAS.COM.MX,WWW.MODATELAS.COM,Fijo,21.883276,-102.296714,2010-07
3,6281210,01001462210000021001021645S9,0552 SUBURBIA AGUASCALIENTES,SUBURBIA S DE RL DE CV,462210,Comercio al por menor en tiendas departamentales,51 a 100 personas,AVENIDA,INDEPENDENCIA,CALLE,MONTES HIMALAYA,BOULEVARD,LUIS DONALDO COLOSIO,BOULEVARD,BOULEVARD A ZACATECAS,2351,D,NaN,NaN,NaN,NaN,COLONIA,TROJES DE ALONSO,NaN,NaN,NaN,20120,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,229,2,NaN,CONTACTO@SUBURBIA.COM.MX,WWW.SUBURBIA.COM.MX,Fijo,21.924398,-102.297908,2010-07
4,8341996,01001462210000354000021645S4,0679 SUBURBIA MAHATMA GANDHI,SUBURBIA S DE RL DE CV,462210,Comercio al por menor en tiendas departamentales,51 a 100 personas,AVENIDA,AGUASCALIENTES,PROLONGACION,PASEO DE LA ASUNCIÓN,AVENIDA,MAHATMA GHANDI,CALLE,FRAY JUNÍPERO SERRA,117,NaN,DESAROLLO ESPECIAL VILLA ASUNCION VILLA JARDIN,PLANTA BAJA,117.0,NaN,COLONIA,VILLA JARDIN,NaN,NaN,NaN,20235,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,869,42,NaN,CAPRISTO.SUBURBIA@GMAIL.COM,WWW.SUBURBIA.COM,Fijo,21.859739,-102.295314,2019-11


ZIP: denue_00_46321-46531_0525_csv.zip
CSV: conjunto_de_datos/denue_inegi_46321-46531_.csv
Número de columnas: 42


,id,clee,nom_estab,raz_social,codigo_act,nombre_act,per_ocu,tipo_vial,nom_vial,tipo_v_e_1,nom_v_e_1,tipo_v_e_2,nom_v_e_2,tipo_v_e_3,nom_v_e_3,numero_ext,letra_ext,edificio,edificio_e,numero_int,letra_int,tipo_asent,nomb_asent,tipoCenCom,nom_CenCom,num_local,cod_postal,cve_ent,entidad,cve_mun,municipio,cve_loc,localidad,ageb,manzana,telefono,correoelec,www,tipoUniEco,latitud,longitud,fecha_alta
0,6282399,01001463310003231000061157S8,015 SKX ALTARIA,MANHATTAN SKMX S DE RL DE CV,465215,Comercio al por menor de artículos y aparatos ...,11 a 30 personas,BOULEVARD,ZACATECAS NTE.,BOULEVARD,LUIS DONALDO COLOSIO,AVENIDA,AGUASCALIENTES,CALLE,ARTICULO 1,849,NaN,NaN,NaN,NaN,NaN,COLONIA,TROJES DE ALONSO,CENTRO Y PLAZA COMERCIAL,ALTARIA,2025,20116.0,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1320,66,NaN,RMEJIA@CHARLY.COM,WWW.SKECHERS.COM.MX,Fijo,21.924925,-102.290166,2010-07
1,6282398,01001463310003081000005304S7,037 CHARLY PARIAN,INTER TENIS SA DE CV,465215,Comercio al por menor de artículos y aparatos ...,11 a 30 personas,PEATONAL,ALLENDE ORIENTE,CALLE,BENITO JUAREZ,CALLE,REFORMA AGRARIA,CALLE,GUADALUPE VICTORIA,76,NaN,NaN,NaN,NaN,NaN,COLONIA,CENTRO,CENTRO Y PLAZA COMERCIAL,EL PARIAN,76,20000.0,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,638,9,NaN,RMEJIA@CHARLY.COM,WWW.CHARLY.COM,Fijo,21.882893,-102.295494,2010-07
2,6281814,01001463211011603001007374S8,1011 TIENDAS ELECZION,GRUPO ELECZION S DE RL DE CV,463211,"Comercio al por menor de ropa, excepto de bebé...",11 a 30 personas,CALLE,5 DE MAYO,CALLE,PLAZA PRINCIPAL NORTE,PEATONAL,ALLENDE ORIENTE,PEATONAL,BENITO JUÁREZ,115,NaN,NaN,NaN,NaN,NaN,COLONIA,CENTRO,NaN,NaN,NaN,20000.0,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,638,13,NaN,STORE1011@GRUPOVISION.COM.MX,WWW.ELECZION.COM,Fijo,21.881665,-102.296788,2014-12
3,6282051,01001465212000842002003732S1,133 ALTARIA AGUASCALIENTES,DISTRIBUIDORA JUGUETRON SA DE CV,465212,Comercio al por menor de juguetes,11 a 30 personas,BOULEVARD,A ZACATECAS NORTE,AVENIDA,AGUASCALIENTES NTE,OTRO(ESPECIFIQUE),NINGUNO,OTRO(ESPECIFIQUE),NINGUNO,849,NaN,NaN,NaN,NaN,NaN,COLONIA,TROJES DE ALONSO,CENTRO Y PLAZA COMERCIAL,ALTARIA,2011-2012,NaN,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1320,66,4.499933e+09,AGUASCALIENTES33@JUGUETRON.COM,WWW.JUGUETRON.MX,Fijo,21.922882,-102.290143,2010-07
4,6281375,01001463211002851002003958S4,3201 MILANO,MILANO OPERADORA SA DE CV,463211,"Comercio al por menor de ropa, excepto de bebé...",31 a 50 personas,AVENIDA,JUÁREZ,PASAJE,SAN ANTONIO,CALLE,RIVERO Y GUTIÉRREZ,PASAJE,ORTEGA,314,NaN,NaN,NaN,NaN,NaN,COLONIA,CENTRO,NaN,NaN,NaN,NaN,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,515,45,NaN,ATENCION.MILANO@MILANO.COM,WWW.MILANO.COM,Fijo,21.883797,-102.296351,2010-07


ZIP: denue_00_46591-46911_0525_csv.zip
CSV: conjunto_de_datos/denue_inegi_46591-46911_.csv
Número de columnas: 42


,id,clee,nom_estab,raz_social,codigo_act,nombre_act,per_ocu,tipo_vial,nom_vial,tipo_v_e_1,nom_v_e_1,tipo_v_e_2,nom_v_e_2,tipo_v_e_3,nom_v_e_3,numero_ext,letra_ext,edificio,edificio_e,numero_int,letra_int,tipo_asent,nomb_asent,tipoCenCom,nom_CenCom,num_local,cod_postal,cve_ent,entidad,cve_mun,municipio,cve_loc,localidad,ageb,manzana,telefono,correoelec,www,tipoUniEco,latitud,longitud,fecha_alta
0,9432795,01001466112001462000003387S8,115 MEGA PLAZA VELARIA MALL AGS,NUEVA ELEKTRA DEL MILENIO SA DE CV,466112,Comercio al por menor de electrodomésticos men...,6 a 10 personas,AVENIDA,AGUASCALIENTES PONIENTE,AVENIDA,AVENIDA DE LOS MAESTROS,CALLE,MÁLAGA,OTRO(ESPECIFIQUE),NINGUNO,402,NaN,NaN,NaN,NaN,NaN,COLONIA,ESPAÑA,NaN,NaN,NaN,20210,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1373,29,NaN,CONTACTO@ELEKTRA.MX,WWW.ELEKTRA.MX,Fijo,21.859367,-102.315435,2024-05
1,6281167,01001466112000371001003387S2,1342 EKT AGUASCALIENTES 2 ASUNCION,NUEVA ELEKTRA DEL MILENIO SA DE CV,466112,Comercio al por menor de electrodomésticos men...,0 a 5 personas,AVENIDA,MAHATMA GANDHI,CALLE,VALENTE QUINTANA,CALLE,ABRAHAM GONZÁLEZ,BOULEVARD,JOSÉ MARÍA CHÁVEZ,319,NaN,NaN,NaN,NaN,NaN,FRACCIONAMIENTO,PILAR BLANCO,CENTRO Y PLAZA COMERCIAL,CENTRO COMERCIAL PLAZA VILLA ASUNCION,SN,20289,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1458,27,NaN,CONTACTO@ELEKTRA.MX,WWW.ELEKTRA.COM.MX,Fijo,21.850052,-102.293881,2010-07
2,6281352,01001466112000572000003387S5,1504 EKT AGUASCALIENTES 1 ALLENDE,NUEVA ELEKTRA DEL MILENIO SA DE CV,466112,Comercio al por menor de electrodomésticos men...,0 a 5 personas,PEATONAL,ALLENDE ORIENTE,AVENIDA,FRANCISCO I MADERO,CALLE,5 DE MAYO,CALLE,LICENCIADO BENITO JUAREZ,117,NaN,NaN,NaN,NaN,NaN,COLONIA,CENTRO,NaN,NaN,NaN,20000,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,638,13,NaN,CONTACTO@ELEKTRA.MX,WWW.ELEKTRA.COM.MX,Fijo,21.882179,-102.296906,2010-07
3,6281967,01007466112000022001003387S8,2212 EKT AGS RINCON DE ROMOS,NUEVA ELEKTRA DEL MILENIO SA DE CV,466112,Comercio al por menor de electrodomésticos men...,0 a 5 personas,AVENIDA,MORELOS,CALLE,HEROICO COLEGIO MILITAR PONIENTE,CALLE,MOTOLINÍA PONIENTE,AVENIDA,LIBERTAD,413,NaN,NaN,NaN,NaN,NaN,COLONIA,RINCON DE ROMOS,NaN,NaN,NaN,20400,1,Aguascalientes,7,Rincón de Romos,1,Rincón de Romos ...,59,31,NaN,CONTACTO@ELEKTRA.MX,WWW.ELEKTRA.COM.MX,Fijo,22.232380,-102.320610,2010-07
4,9441416,01006466112000082000003387S4,2780 MEGA PABELLON DE ARTEAGA,NUEVA ELEKTRA DEL MILENIO SA DE CV,466112,Comercio al por menor de electrodomésticos men...,6 a 10 personas,AVENIDA,PLUTARCO ELÍAS CALLES,AVENIDA,HEROICO COLEGIO MILITAR,AVENIDA,PINO SUÁREZ,AVENIDA,GLORIETA,45,NaN,NaN,NaN,NaN,NaN,COLONIA,PABELLON DE ARTEAGA CENTRO,NaN,NaN,NaN,20670,1,Aguascalientes,6,Pabellón de Arteaga,1,Pabellón de Arteaga ...,85,14,NaN,CONTACTO@ELEKTRA.MX,WWW.ELEKTRA.MX,Fijo,22.147735,-102.277213,2024-05


### 2.3 Validación de consistencia del esquema DENUE 2025

Se comprueba que los cuatro archivos DENUE 2025 tengan las mismas variables y el mismo orden de columnas antes de integrarlos.

Esta validación permite determinar si los archivos son estructuralmente compatibles.

In [6]:
# Tomar el primer archivo como esquema de referencia
primer_zip_2025 = list(muestras_denue_2025.keys())[0]

columnas_referencia_2025 = list(
    muestras_denue_2025[primer_zip_2025].columns
)

resumen_esquema_2025 = []

for nombre_zip, muestra in muestras_denue_2025.items():

    mismas_columnas = (
        list(muestra.columns) == columnas_referencia_2025
    )

    resumen_esquema_2025.append({
        "archivo": nombre_zip,
        "numero_columnas": muestra.shape[1],
        "mismo_esquema": mismas_columnas
    })

resumen_esquema_2025 = pd.DataFrame(
    resumen_esquema_2025
)

display(resumen_esquema_2025)

,archivo,numero_columnas,mismo_esquema
0,denue_00_46111_0525_csv.zip,42,True
1,denue_00_46112-46311_0525_csv.zip,42,True
2,denue_00_46321-46531_0525_csv.zip,42,True
3,denue_00_46591-46911_0525_csv.zip,42,True


In [7]:
############### 2.4 Validar las variables necesarias
columnas_clave_2025 = [
    "id",
    "codigo_act",
    "nombre_act",
    "per_ocu",
    "cve_ent",
    "entidad",
    "cve_mun",
    "municipio"
]

validacion_columnas_2025 = []

for nombre_zip, muestra in muestras_denue_2025.items():

    registro = {"archivo": nombre_zip}

    for columna in columnas_clave_2025:
        registro[columna] = columna in muestra.columns

    validacion_columnas_2025.append(registro)

validacion_columnas_2025 = pd.DataFrame(
    validacion_columnas_2025
)

display(validacion_columnas_2025)

,archivo,id,codigo_act,nombre_act,per_ocu,cve_ent,entidad,cve_mun,municipio
0,denue_00_46111_0525_csv.zip,True,True,True,True,True,True,True,True
1,denue_00_46112-46311_0525_csv.zip,True,True,True,True,True,True,True,True
2,denue_00_46321-46531_0525_csv.zip,True,True,True,True,True,True,True,True
3,denue_00_46591-46911_0525_csv.zip,True,True,True,True,True,True,True,True


### 2.5 Revisión del diccionario oficial de datos DENUE 2025

Se consulta el diccionario de datos incluido en los paquetes DENUE 2025 con el objetivo de:

- identificar formalmente las variables disponibles;
- documentar su descripción oficial;
- verificar la nueva estructura de 42 variables;
- identificar las diferencias respecto de DENUE 2020;
- confirmar las definiciones de las variables que serán utilizadas en el proyecto.

La comparación entre las ediciones 2020 y 2025 se realizará únicamente después de revisar la documentación incluida en ambas fuentes.

In [8]:
# Utilizar el primer ZIP DENUE 2025 como referencia
archivo_zip_ref_2025 = archivos_zip_2025[0]

with zipfile.ZipFile(archivo_zip_ref_2025, "r") as z:

    archivos_diccionario_2025 = [
        nombre for nombre in z.namelist()
        if nombre.startswith("diccionario_de_datos/")
        and nombre.endswith(".csv")
    ]

    print("Archivos de diccionario encontrados:")
    for nombre in archivos_diccionario_2025:
        print(nombre)

    if len(archivos_diccionario_2025) == 1:

        diccionario_csv_2025 = archivos_diccionario_2025[0]

        with z.open(diccionario_csv_2025) as archivo:
            diccionario_denue_2025 = pd.read_csv(
                archivo,
                encoding="latin-1",
                header=1
            )

        # Limpiar espacios en encabezados
        diccionario_denue_2025.columns = (
            diccionario_denue_2025.columns.str.strip()
        )

        print("\nDimensiones del diccionario:")
        print(diccionario_denue_2025.shape)

        print("\nColumnas del diccionario:")
        print(diccionario_denue_2025.columns.tolist())

        display(diccionario_denue_2025.head(15))

Archivos de diccionario encontrados:
diccionario_de_datos/denue_diccionario_de_datos.csv

Dimensiones del diccionario:
(42, 5)

Columnas del diccionario:
['Nombre del Atributo en csv', 'Nombre del Atributo en DBF', 'Tipo de dato', 'Longitud', 'Descripción']


,Nombre del Atributo en csv,Nombre del Atributo en DBF,Tipo de dato,Longitud,Descripción
0,id,id,numérico,10,"Número de identificación del DENUE, es una cl..."
1,nom_estab,nom_estab,alfanumérico,150,Es el nombre comercial o nombre exterior con e...
2,raz_social,raz_social,alfanumérico,250,Es la forma con que está legalmente constituid...
3,codigo_act,codigo_act,alfanumérico,6,La clasificación de las actividades desarrolla...
4,nombre_act,nombre_act,alfanumérico,250,Nombre del código de actividad conforme al SCI...
5,per_ocu,per_ocu,alfanumérico,20,Comprende al personal contratado directamente ...
6,tipo_vial,tipo_vial,alfanumérico,40,Es la superficie del terreno destinada para el...
7,nom_vial,nom_vial,alfanumérico,150,Es el sustantivo propio con el cual se identif...
8,tipo_v_e_1,tipo_v_e_1,alfanumérico,40,Es la superficie del terreno destinada para el...
9,nom_v_e_1,nom_v_e_1,alfanumérico,150,Es el sustantivo propio con el cual se identif...


In [9]:
##2.6 Verificar clee en el diccionario 2025
clee_diccionario = diccionario_denue_2025[
    diccionario_denue_2025["Nombre del Atributo en csv"]
    .str.lower()
    .eq("clee")
]

print("Registros encontrados para clee:")
display(clee_diccionario)

Registros encontrados para clee:


,Nombre del Atributo en csv,Nombre del Atributo en DBF,Tipo de dato,Longitud,Descripción
41,clee,clee,alfanumérico,40,Es la llave única de identificación estadístic...


### 2.7 Comparación estructural DENUE 2020 vs. DENUE 2025

Se comparan directamente los nombres de las variables presentes en ambas ediciones de DENUE.

El objetivo es identificar:

- variables comunes;
- variables exclusivas de 2020;
- variables exclusivas de 2025;
- diferencias estructurales que deban considerarse antes de comparar ambos periodos.

In [10]:
# Ruta DENUE 2020
DENUE_2020_DIR = PROJECT_ROOT / "data" / "raw" / "denue" / "2020"

# Primer ZIP 2020
archivo_2020_ref = sorted(DENUE_2020_DIR.glob("*.zip"))[0]

with zipfile.ZipFile(archivo_2020_ref, "r") as z:

    csv_2020 = [
        nombre for nombre in z.namelist()
        if nombre.startswith("conjunto_de_datos/")
        and nombre.endswith(".csv")
    ][0]

    with z.open(csv_2020) as archivo:
        columnas_2020 = pd.read_csv(
            archivo,
            encoding="latin-1",
            nrows=0
        ).columns.tolist()


# Columnas 2025 obtenidas de la muestra
columnas_2025 = columnas_referencia_2025


print("Columnas DENUE 2020:", len(columnas_2020))
print("Columnas DENUE 2025:", len(columnas_2025))

Columnas DENUE 2020: 41
Columnas DENUE 2025: 42


In [11]:
set_2020 = set(columnas_2020)
set_2025 = set(columnas_2025)

solo_2020 = sorted(set_2020 - set_2025)
solo_2025 = sorted(set_2025 - set_2020)
comunes = sorted(set_2020 & set_2025)

print("Variables comunes:", len(comunes))
print("Solo en 2020:", solo_2020)
print("Solo en 2025:", solo_2025)

Variables comunes: 41
Solo en 2020: []
Solo en 2025: ['clee']


### 2.8 Conclusión de comparabilidad estructural DENUE 2020–2025

La comparación de los esquemas de las ediciones DENUE 2020 y DENUE 2025 muestra una alta compatibilidad estructural.

Resultados:

- DENUE 2020 contiene 41 variables.
- DENUE 2025 contiene 42 variables.
- Las 41 variables presentes en DENUE 2020 también están disponibles en DENUE 2025.
- DENUE 2025 incorpora una variable adicional denominada `clee`.
- No existen variables exclusivas de DENUE 2020.
- Las variables necesarias para el proyecto (`id`, `codigo_act`, `nombre_act`, `per_ocu`, `cve_ent`, `entidad`, `cve_mun` y `municipio`) están disponibles en ambas ediciones.

Por tanto, ambos conjuntos de datos son estructuralmente compatibles para la construcción de las bases municipales del proyecto.

La compatibilidad estructural no implica todavía equivalencia completa de las clasificaciones económicas entre ambos periodos; la comparación de actividades se realizará a un nivel agregado de comercio al por menor para reducir el efecto de cambios en la clasificación SCIAN.

### 2.9 Conteo de registros por archivo DENUE 2025

Se contabiliza el número de establecimientos contenidos en cada uno de los cuatro archivos DENUE 2025 correspondientes al comercio al por menor.

La lectura se realiza por bloques para mantener un uso controlado de memoria y obtener el volumen total de registros que posteriormente serán agregados a nivel municipal.

In [12]:
resumen_registros_2025 = []

for archivo_zip in archivos_zip_2025:

    print(f"Procesando: {archivo_zip.name}")

    with zipfile.ZipFile(archivo_zip, "r") as z:

        archivos_datos = [
            nombre for nombre in z.namelist()
            if nombre.startswith("conjunto_de_datos/")
            and nombre.endswith(".csv")
        ]

        csv_principal = archivos_datos[0]

        total_registros = 0

        with z.open(csv_principal) as archivo_csv:

            for chunk in pd.read_csv(
                archivo_csv,
                encoding="latin-1",
                usecols=["id"],
                chunksize=200_000
            ):

                total_registros += len(chunk)

        resumen_registros_2025.append({
            "archivo": archivo_zip.name,
            "registros": total_registros
        })

        print(
            f"Registros encontrados: "
            f"{total_registros:,}\n"
        )

resumen_registros_2025 = pd.DataFrame(
    resumen_registros_2025
)

display(resumen_registros_2025)

Procesando: denue_00_46111_0525_csv.zip
Registros encontrados: 672,075

Procesando: denue_00_46112-46311_0525_csv.zip
Registros encontrados: 589,295

Procesando: denue_00_46321-46531_0525_csv.zip
Registros encontrados: 639,384

Procesando: denue_00_46591-46911_0525_csv.zip
Registros encontrados: 635,584



,archivo,registros
0,denue_00_46111_0525_csv.zip,672075
1,denue_00_46112-46311_0525_csv.zip,589295
2,denue_00_46321-46531_0525_csv.zip,639384
3,denue_00_46591-46911_0525_csv.zip,635584


In [13]:
total_establecimientos_2025 = (
    resumen_registros_2025["registros"].sum()
)

print(
    f"Total de establecimientos de comercio al por menor "
    f"considerados en DENUE 2025: "
    f"{total_establecimientos_2025:,}"
)

Total de establecimientos de comercio al por menor considerados en DENUE 2025: 2,536,338


### 2.10 Validación de unicidad del identificador DENUE 2025

Antes de integrar los cuatro archivos DENUE 2025 se verifica la calidad del identificador `id`.

Se comprobará:

- existencia de valores nulos en `id`;
- identificadores duplicados dentro de cada archivo;
- identificadores repetidos entre diferentes archivos;
- número total de identificadores únicos.

Este control permite evitar la doble contabilización de establecimientos durante la posterior agregación municipal.

In [14]:
resumen_ids_2025 = []

# Conjunto acumulado de identificadores
ids_globales_2025 = set()

for archivo_zip in archivos_zip_2025:

    print(f"Validando: {archivo_zip.name}")

    ids_archivo = set()
    ids_duplicados_internos = set()

    total_registros = 0
    total_nulos_id = 0

    with zipfile.ZipFile(archivo_zip, "r") as z:

        archivos_datos = [
            nombre for nombre in z.namelist()
            if nombre.startswith("conjunto_de_datos/")
            and nombre.endswith(".csv")
        ]

        csv_principal = archivos_datos[0]

        with z.open(csv_principal) as archivo_csv:

            for chunk in pd.read_csv(
                archivo_csv,
                encoding="latin-1",
                usecols=["id"],
                chunksize=200_000
            ):

                total_registros += len(chunk)

                ids = pd.to_numeric(
                    chunk["id"],
                    errors="coerce"
                )

                total_nulos_id += ids.isna().sum()

                ids_validos = (
                    ids
                    .dropna()
                    .astype("int64")
                )

                # Duplicados dentro del mismo chunk
                repetidos_chunk = set(
                    ids_validos[
                        ids_validos.duplicated(keep=False)
                    ].tolist()
                )

                # Duplicados contra chunks anteriores
                ids_chunk = set(ids_validos.tolist())

                repetidos_previos = (
                    ids_chunk.intersection(ids_archivo)
                )

                ids_duplicados_internos.update(
                    repetidos_chunk
                )

                ids_duplicados_internos.update(
                    repetidos_previos
                )

                ids_archivo.update(ids_chunk)

    # IDs que también existen en alguno de los archivos anteriores
    ids_repetidos_entre_archivos = (
        ids_archivo.intersection(ids_globales_2025)
    )

    resumen_ids_2025.append({
        "archivo": archivo_zip.name,
        "registros": total_registros,
        "id_nulos": total_nulos_id,
        "id_unicos_archivo": len(ids_archivo),
        "id_duplicados_internos": len(
            ids_duplicados_internos
        ),
        "id_repetidos_entre_archivos": len(
            ids_repetidos_entre_archivos
        )
    })

    ids_globales_2025.update(ids_archivo)

    print(f"IDs únicos: {len(ids_archivo):,}")
    print(
        f"IDs duplicados internos: "
        f"{len(ids_duplicados_internos):,}"
    )
    print(
        f"IDs compartidos con otros archivos: "
        f"{len(ids_repetidos_entre_archivos):,}"
    )
    print()

resumen_ids_2025 = pd.DataFrame(
    resumen_ids_2025
)

display(resumen_ids_2025)

Validando: denue_00_46111_0525_csv.zip
IDs únicos: 672,075
IDs duplicados internos: 0
IDs compartidos con otros archivos: 0

Validando: denue_00_46112-46311_0525_csv.zip
IDs únicos: 589,295
IDs duplicados internos: 0
IDs compartidos con otros archivos: 0

Validando: denue_00_46321-46531_0525_csv.zip
IDs únicos: 639,384
IDs duplicados internos: 0
IDs compartidos con otros archivos: 0

Validando: denue_00_46591-46911_0525_csv.zip
IDs únicos: 635,584
IDs duplicados internos: 0
IDs compartidos con otros archivos: 0



,archivo,registros,id_nulos,id_unicos_archivo,id_duplicados_internos,id_repetidos_entre_archivos
0,denue_00_46111_0525_csv.zip,672075,0,672075,0,0
1,denue_00_46112-46311_0525_csv.zip,589295,0,589295,0,0
2,denue_00_46321-46531_0525_csv.zip,639384,0,639384,0,0
3,denue_00_46591-46911_0525_csv.zip,635584,0,635584,0,0


### 2.11 Validación de claves geográficas y cobertura municipal DENUE 2025

Se valida la calidad de las claves de entidad y municipio antes de agregar los establecimientos a nivel municipal.

Se verifican:

- valores faltantes en `cve_ent` y `cve_mun`;
- formato de las claves;
- construcción de `CVEGEO`;
- número de municipios presentes en cada archivo;
- cobertura municipal total de DENUE 2025.

Posteriormente, esta cobertura será comparada con DENUE 2020 para identificar cambios territoriales entre ambos periodos.

In [15]:
resumen_geografico_2025 = []
catalogos_municipales_2025 = []

for archivo_zip in archivos_zip_2025:

    print(f"Validando: {archivo_zip.name}")

    total_registros = 0
    nulos_ent = 0
    nulos_mun = 0
    claves_invalidas = 0

    municipios_archivo = set()

    with zipfile.ZipFile(archivo_zip, "r") as z:

        archivos_datos = [
            nombre for nombre in z.namelist()
            if nombre.startswith("conjunto_de_datos/")
            and nombre.endswith(".csv")
        ]

        csv_principal = archivos_datos[0]

        with z.open(csv_principal) as archivo_csv:

            for chunk in pd.read_csv(
                archivo_csv,
                encoding="latin-1",
                usecols=[
                    "cve_ent",
                    "entidad",
                    "cve_mun",
                    "municipio"
                ],
                dtype="string",
                chunksize=200_000
            ):

                total_registros += len(chunk)

                chunk["cve_ent"] = chunk["cve_ent"].str.strip()
                chunk["cve_mun"] = chunk["cve_mun"].str.strip()

                nulos_ent += chunk["cve_ent"].isna().sum()
                nulos_mun += chunk["cve_mun"].isna().sum()

                chunk["cve_ent"] = chunk["cve_ent"].str.zfill(2)
                chunk["cve_mun"] = chunk["cve_mun"].str.zfill(3)

                chunk["CVEGEO"] = (
                    chunk["cve_ent"]
                    + chunk["cve_mun"]
                )

                valida = (
                    chunk["CVEGEO"].str.len().eq(5)
                    & chunk["CVEGEO"].str.isnumeric()
                )

                claves_invalidas += (
                    ~valida.fillna(False)
                ).sum()

                municipios_archivo.update(
                    chunk.loc[
                        valida,
                        "CVEGEO"
                    ].dropna()
                )

                catalogo_chunk = (
                    chunk.loc[
                        valida,
                        ["CVEGEO", "entidad", "municipio"]
                    ]
                    .drop_duplicates()
                )

                catalogos_municipales_2025.append(
                    catalogo_chunk
                )

    resumen_geografico_2025.append({
        "archivo": archivo_zip.name,
        "registros": total_registros,
        "nulos_cve_ent": nulos_ent,
        "nulos_cve_mun": nulos_mun,
        "claves_invalidas": claves_invalidas,
        "municipios_unicos": len(municipios_archivo)
    })

    print(f"Municipios únicos: {len(municipios_archivo):,}")
    print(f"Nulos cve_ent: {nulos_ent:,}")
    print(f"Nulos cve_mun: {nulos_mun:,}")
    print(f"Claves inválidas: {claves_invalidas:,}")
    print()

resumen_geografico_2025 = pd.DataFrame(
    resumen_geografico_2025
)

display(resumen_geografico_2025)

Validando: denue_00_46111_0525_csv.zip
Municipios únicos: 2,477
Nulos cve_ent: 0
Nulos cve_mun: 0
Claves inválidas: 0

Validando: denue_00_46112-46311_0525_csv.zip
Municipios únicos: 2,421
Nulos cve_ent: 0
Nulos cve_mun: 0
Claves inválidas: 0

Validando: denue_00_46321-46531_0525_csv.zip
Municipios únicos: 2,403
Nulos cve_ent: 0
Nulos cve_mun: 0
Claves inválidas: 0

Validando: denue_00_46591-46911_0525_csv.zip
Municipios únicos: 2,371
Nulos cve_ent: 0
Nulos cve_mun: 0
Claves inválidas: 0



,archivo,registros,nulos_cve_ent,nulos_cve_mun,claves_invalidas,municipios_unicos
0,denue_00_46111_0525_csv.zip,672075,0,0,0,2477
1,denue_00_46112-46311_0525_csv.zip,589295,0,0,0,2421
2,denue_00_46321-46531_0525_csv.zip,639384,0,0,0,2403
3,denue_00_46591-46911_0525_csv.zip,635584,0,0,0,2371


In [16]:
catalogo_municipal_2025 = (
    pd.concat(
        catalogos_municipales_2025,
        ignore_index=True
    )
    .drop_duplicates()
)

municipios_unicos_2025 = (
    catalogo_municipal_2025["CVEGEO"].nunique()
)

print(
    f"Municipios únicos cubiertos por DENUE retail 2025: "
    f"{municipios_unicos_2025:,}"
)

Municipios únicos cubiertos por DENUE retail 2025: 2,477


2.12 Consistencia de nombres en DENUE 2025
### 2.12 Validación de consistencia entre CVEGEO y municipio — DENUE 2025

Se verifica que cada clave geográfica municipal esté asociada con una sola entidad federativa y un solo nombre de municipio.

Este control permite confirmar la consistencia del catálogo geográfico antes de comparar DENUE 2025 con DENUE 2020.

In [17]:
consistencia_cvegeo_2025 = (
    catalogo_municipal_2025
    .groupby("CVEGEO")
    .agg(
        entidades_distintas=("entidad", "nunique"),
        municipios_distintos=("municipio", "nunique")
    )
    .reset_index()
)

cvegeo_inconsistentes_2025 = consistencia_cvegeo_2025[
    (consistencia_cvegeo_2025["entidades_distintas"] > 1)
    | (consistencia_cvegeo_2025["municipios_distintos"] > 1)
]

print(
    "CVEGEO con inconsistencias de nombre:",
    len(cvegeo_inconsistentes_2025)
)

display(cvegeo_inconsistentes_2025.head(20))

CVEGEO con inconsistencias de nombre: 0


,CVEGEO,entidades_distintas,municipios_distintos


### 2.13 Comparación de cobertura municipal DENUE 2020–2025

Se comparan las claves municipales presentes en DENUE 2020 y DENUE 2025.

El objetivo es identificar cambios en la cobertura territorial antes de calcular la variación de la densidad comercial.

No se asumirá que la diferencia en el número total de municipios corresponde automáticamente a municipios nuevos; la comparación se realizará directamente mediante `CVEGEO`.

In [18]:
####Primero cargamos cargo la base municipal 2020 ya procesada:
archivo_denue_2020 = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "denue_2020_municipal.csv"
)

denue_2020_municipal = pd.read_csv(
    archivo_denue_2020,
    dtype={"CVEGEO": "string"}
)

print("Municipios DENUE 2020:", len(denue_2020_municipal))

Municipios DENUE 2020: 2465


In [19]:
###Ahora comparo conjuntos
municipios_2020 = set(
    denue_2020_municipal["CVEGEO"]
)

municipios_2025 = set(
    catalogo_municipal_2025["CVEGEO"]
)

comunes_2020_2025 = municipios_2020 & municipios_2025
solo_2020 = municipios_2020 - municipios_2025
solo_2025 = municipios_2025 - municipios_2020

print(f"Municipios 2020: {len(municipios_2020):,}")
print(f"Municipios 2025: {len(municipios_2025):,}")
print(f"Municipios comunes: {len(comunes_2020_2025):,}")
print(f"Solo en 2020: {len(solo_2020):,}")
print(f"Solo en 2025: {len(solo_2025):,}")

Municipios 2020: 2,465
Municipios 2025: 2,477
Municipios comunes: 2,465
Solo en 2020: 0
Solo en 2025: 12


In [20]:
#######32.14 Identificar exactamente cuáles son
municipios_solo_2025 = (
    catalogo_municipal_2025[
        catalogo_municipal_2025["CVEGEO"].isin(solo_2025)
    ]
    [["CVEGEO", "entidad", "municipio"]]
    .drop_duplicates()
    .sort_values("CVEGEO")
    .reset_index(drop=True)
)

display(municipios_solo_2025)

,CVEGEO,entidad,municipio
0,02006,Baja California,San Quintín
1,02007,Baja California,San Felipe ...
2,04012,Campeche,Seybaplaya
3,04013,Campeche,Dzitbalché ...
4,12082,Guerrero,Las Vigas ...
5,12083,Guerrero,Ñuu Savi ...
6,12084,Guerrero,Santa Cruz del Rincón ...
7,12085,Guerrero,San Nicolás ...
8,17036,Morelos,Hueyapan
9,24059,San Luis Potosí,Villa de Pozos SLP


In [21]:
#############si hubiera alguno que apareciera únicamente en 2020
municipios_solo_2020 = (
    denue_2020_municipal[
        denue_2020_municipal["CVEGEO"].isin(solo_2020)
    ]
    [["CVEGEO", "entidad", "municipio"]]
    .drop_duplicates()
    .sort_values("CVEGEO")
    .reset_index(drop=True)
)

display(municipios_solo_2020)

,CVEGEO,entidad,municipio


### 2.15 Conclusión de Data Understanding — DENUE 2025

La revisión de los cuatro paquetes DENUE 2025 correspondientes al comercio al por menor permitió confirmar la calidad estructural y geográfica de los datos.

Principales resultados:

- Se localizaron correctamente los cuatro archivos esperados.
- Los cuatro archivos presentan una estructura común de 42 variables.
- Las ocho variables requeridas para el proyecto están disponibles en todos los archivos.
- DENUE 2025 contiene una variable adicional denominada `clee` respecto de DENUE 2020.
- Las 41 variables existentes en DENUE 2020 también están presentes en DENUE 2025.
- Se identificaron 2,536,338 registros de establecimientos de comercio al por menor.
- No se detectaron identificadores `id` nulos o duplicados.
- No existen identificadores compartidos entre los cuatro archivos.
- No se encontraron valores faltantes en las claves de entidad y municipio.
- No se detectaron claves geográficas inválidas.
- DENUE 2025 presenta cobertura en 2,477 municipios.
- No se encontraron inconsistencias entre `CVEGEO` y los nombres municipales.
- De las 2,477 claves municipales observadas en 2025, 2,465 también se encuentran en DENUE 2020 y 12 aparecen exclusivamente en 2025.

La identificación de claves municipales comunes no implica por sí sola equivalencia territorial completa. Antes de construir la variable de crecimiento comercial 2020–2025 se revisarán los municipios cuya delimitación territorial cambió durante el periodo.

##############################################################################################################################################################

# Fase 3 — Data Preparation
## Construcción de la base municipal DENUE 2025

La unidad original de DENUE es el establecimiento económico, mientras que la unidad de análisis del proyecto es el municipio.

Por esta razón, los establecimientos de comercio al por menor se agregarán utilizando la clave municipal `CVEGEO`.

El resultado será una base con una fila por municipio y las siguientes variables:

- `CVEGEO`
- entidad federativa
- municipio
- `EST_RETAIL_2025`

La variable `EST_RETAIL_2025` representa el número de establecimientos de comercio al por menor registrados en DENUE 2025 para cada municipio.

In [22]:
#####3.1 Agregación municipal
agregados_municipales_2025 = []
catalogos_nombres_2025 = []

for archivo_zip in archivos_zip_2025:

    print(f"Procesando: {archivo_zip.name}")

    with zipfile.ZipFile(archivo_zip, "r") as z:

        archivos_datos = [
            nombre for nombre in z.namelist()
            if nombre.startswith("conjunto_de_datos/")
            and nombre.endswith(".csv")
        ]

        csv_principal = archivos_datos[0]

        with z.open(csv_principal) as archivo_csv:

            for chunk in pd.read_csv(
                archivo_csv,
                encoding="latin-1",
                usecols=[
                    "cve_ent",
                    "entidad",
                    "cve_mun",
                    "municipio"
                ],
                dtype="string",
                chunksize=200_000
            ):

                # Normalizar claves
                chunk["cve_ent"] = (
                    chunk["cve_ent"]
                    .str.strip()
                    .str.zfill(2)
                )

                chunk["cve_mun"] = (
                    chunk["cve_mun"]
                    .str.strip()
                    .str.zfill(3)
                )

                # Construir CVEGEO
                chunk["CVEGEO"] = (
                    chunk["cve_ent"]
                    + chunk["cve_mun"]
                )

                # Conteo parcial por municipio
                agregado_chunk = (
                    chunk
                    .groupby("CVEGEO", as_index=False)
                    .size()
                    .rename(
                        columns={
                            "size": "EST_RETAIL_2025"
                        }
                    )
                )

                agregados_municipales_2025.append(
                    agregado_chunk
                )

                # Catálogo de nombres municipales
                catalogo_chunk = (
                    chunk[
                        [
                            "CVEGEO",
                            "entidad",
                            "municipio"
                        ]
                    ]
                    .drop_duplicates()
                )

                catalogos_nombres_2025.append(
                    catalogo_chunk
                )

    print("Archivo procesado correctamente.\n")

Procesando: denue_00_46111_0525_csv.zip
Archivo procesado correctamente.

Procesando: denue_00_46112-46311_0525_csv.zip
Archivo procesado correctamente.

Procesando: denue_00_46321-46531_0525_csv.zip
Archivo procesado correctamente.

Procesando: denue_00_46591-46911_0525_csv.zip
Archivo procesado correctamente.



In [23]:
####################3.2 Consolidar los conteos

conteo_municipal_2025 = (
    pd.concat(
        agregados_municipales_2025,
        ignore_index=True
    )
    .groupby(
        "CVEGEO",
        as_index=False
    )["EST_RETAIL_2025"]
    .sum()
)

conteo_municipal_2025["EST_RETAIL_2025"] = (
    conteo_municipal_2025["EST_RETAIL_2025"]
    .astype("int64")
)

display(conteo_municipal_2025.head(10))

,CVEGEO,EST_RETAIL_2025
0,01001,18552
1,01002,579
2,01003,1062
3,01004,256
4,01005,2088
5,01006,987
6,01007,1282
7,01008,237
8,01009,289
9,01010,267


In [24]:
###################3.3 Catálogo municipal

catalogo_nombres_2025 = (
    pd.concat(
        catalogos_nombres_2025,
        ignore_index=True
    )
    .drop_duplicates()
    .sort_values("CVEGEO")
    .reset_index(drop=True)
)

print(
    "Municipios en el catálogo:",
    f"{catalogo_nombres_2025['CVEGEO'].nunique():,}"
)

Municipios en el catálogo: 2,477


In [25]:
################3.4 Construir base municipal
denue_2025_municipal = (
    catalogo_nombres_2025
    .merge(
        conteo_municipal_2025,
        on="CVEGEO",
        how="inner",
        validate="one_to_one"
    )
    .sort_values("CVEGEO")
    .reset_index(drop=True)
)

print(
    "Dimensiones de la base municipal DENUE 2025:"
)

print(denue_2025_municipal.shape)

display(denue_2025_municipal.head(10))

Dimensiones de la base municipal DENUE 2025:
(2477, 4)


,CVEGEO,entidad,municipio,EST_RETAIL_2025
0,01001,Aguascalientes,Aguascalientes,18552
1,01002,Aguascalientes,Asientos,579
2,01003,Aguascalientes,Calvillo,1062
3,01004,Aguascalientes,Cosío,256
4,01005,Aguascalientes,Jesús María,2088
5,01006,Aguascalientes,Pabellón de Arteaga,987
6,01007,Aguascalientes,Rincón de Romos,1282
7,01008,Aguascalientes,San José de Gracia,237
8,01009,Aguascalientes,Tepezalá,289
9,01010,Aguascalientes,El Llano,267


In [26]:
##############3.5 Control de calidad
print("CONTROL DE CALIDAD — DENUE MUNICIPAL 2025")
print("=" * 55)

print(
    "Número de municipios:",
    f"{len(denue_2025_municipal):,}"
)

print(
    "CVEGEO únicos:",
    f"{denue_2025_municipal['CVEGEO'].nunique():,}"
)

print(
    "CVEGEO duplicados:",
    denue_2025_municipal["CVEGEO"].duplicated().sum()
)

print(
    "Valores nulos totales:",
    denue_2025_municipal.isna().sum().sum()
)

print(
    "Municipios con conteo <= 0:",
    (
        denue_2025_municipal["EST_RETAIL_2025"]
        <= 0
    ).sum()
)

print(
    "Total de establecimientos después de agregar:",
    f"{denue_2025_municipal['EST_RETAIL_2025'].sum():,}"
)

CONTROL DE CALIDAD — DENUE MUNICIPAL 2025
Número de municipios: 2,477
CVEGEO únicos: 2,477
CVEGEO duplicados: 0
Valores nulos totales: 0
Municipios con conteo <= 0: 0
Total de establecimientos después de agregar: 2,536,338


In [28]:
###############3.6 Validación automática

assert len(denue_2025_municipal) == 2477, \
    "El número de municipios no coincide."

assert (
    denue_2025_municipal["CVEGEO"].nunique()
    == 2477
), "Existen problemas de unicidad en CVEGEO."

assert (
    denue_2025_municipal["CVEGEO"]
    .duplicated()
    .sum()
    == 0
), "Existen CVEGEO duplicados."

assert (
    denue_2025_municipal
    .isna()
    .sum()
    .sum()
    == 0
), "Existen valores faltantes."

assert (
    denue_2025_municipal["EST_RETAIL_2025"].sum()
    == total_establecimientos_2025
), "El total de establecimientos no coincide."

assert (
    denue_2025_municipal["EST_RETAIL_2025"]
    .gt(0)
    .all()
), "Existen municipios con conteos no positivos."

print("Todas las validaciones fueron superadas correctamente.")

Todas las validaciones fueron superadas correctamente.


### 3.7 Exportación de la base municipal DENUE 2025

Después de superar los controles de calidad, la base municipal DENUE 2025 se guarda como archivo procesado.

El archivo contiene una observación por municipio y la variable `EST_RETAIL_2025`, correspondiente al número de establecimientos de comercio al por menor registrados en DENUE 2025.

Esta base será utilizada posteriormente junto con DENUE 2020 y la población municipal para construir la variación de la densidad comercial entre 2020 y 2025.

In [29]:
# Ruta de salida
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

archivo_salida_2025 = (
    PROCESSED_DIR / "denue_2025_municipal.csv"
)

denue_2025_municipal.to_csv(
    archivo_salida_2025,
    index=False,
    encoding="utf-8-sig"
)

print("Base DENUE 2025 guardada correctamente.")
print(f"Ruta: {archivo_salida_2025}")

Base DENUE 2025 guardada correctamente.
Ruta: c:\Users\nashe\Desktop\Econometría\PROYECTO\atractividad_comercial_Mexico\data\processed\denue_2025_municipal.csv


In [30]:
###Verificación del archivo guardado
denue_2025_verificacion = pd.read_csv(
    archivo_salida_2025,
    dtype={"CVEGEO": "string"}
)

print(
    "Dimensiones:",
    denue_2025_verificacion.shape
)

print(
    "Total de establecimientos:",
    f"{denue_2025_verificacion['EST_RETAIL_2025'].sum():,}"
)

display(
    denue_2025_verificacion.head()
)

Dimensiones: (2477, 4)
Total de establecimientos: 2,536,338


,CVEGEO,entidad,municipio,EST_RETAIL_2025
0,01001,Aguascalientes,Aguascalientes,18552
1,01002,Aguascalientes,Asientos,579
2,01003,Aguascalientes,Calvillo,1062
3,01004,Aguascalientes,Cosío,256
4,01005,Aguascalientes,Jesús María,2088
